# 01 · 데이터와 평가 (방법론의 backbone)

모든 모델이 **같은 평가기 하나**(`src.metrics.evaluate_corpus`)로 채점된다 — 이게 비교의 전제.
정규화(lowercase → 토큰화(하이픈 유지) → Porter stemming)가 전 실험 동일하다.

- **데이터**: [KP20k](https://huggingface.co/datasets/taln-ls2n/kp20k) — train 530,809 / test 20,000
- **PRMU**: 각 gold를 표면 겹침으로 분류 — **P**resent / **R**eordered / **M**ixed / **U**nseen (중요도 아님, 표면 관계)

In [1]:
import sys, json, warnings
from pathlib import Path
import pandas as pd
warnings.filterwarnings("ignore")

# repo 루트 자동 탐색 (results/ 가 있는 곳) — 노트북을 어디서 열든 동작
ROOT = Path.cwd()
while not (ROOT / "results").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))            # src.* import 가능
sys.path.insert(0, str(ROOT / "scripts"))  # 스크립트 모듈 import 가능
print("repo root:", ROOT)

repo root: C:\Users\wodlf\OneDrive\Desktop\kp20k-keyphrase-extraction


In [2]:
# 실제 데이터 로더 (src.data) — 소량 표본만. 오프라인이면 건너뜀.
try:
    from src.data import load_kp20k, plain_doc_text
    sample = list(load_kp20k(["test"], subset_sizes={"test": 50}, seed=42)["test"])
    r = sample[0]
    print("제목 :", r["title"])
    print("초록 :", (r["abstract"] or "")[:180], "...")
    print("gold :", r["keyphrases"])
    print("prmu :", r["prmu"], " ← 각 gold의 표면 관계 (P/R/M/U)")
except Exception as e:
    print("데이터 로드 건너뜀(오프라인 등):", type(e).__name__, e)

제목 : Narrow band region-based active contours and surfaces for 2D and 3D segmentation
초록 : We describe a narrow band region approach for deformable curves and surfaces in the perspective of 2D and 3D image segmentation. Basically, we develop a region energy involving a f ...
gold : ['active contour', 'segmentation', 'deformable model', 'level sets', 'narrow band region energy', 'active surface']
prmu : ['P', 'P', 'P', 'P', 'R', 'R']  ← 각 gold의 표면 관계 (P/R/M/U)


### PRMU 분류와 stemming — 스크립트가 쓰는 바로 그 함수

In [3]:
from src.preprocessing import classify_prmu, stem_tokens

doc = "we propose a convolutional neural network for image classification"
doc_stems = stem_tokens(doc)
for phrase in ["convolutional neural network", "neural networks", "deep learning"]:
    print(f"{phrase:35s} -> {classify_prmu(phrase, '', doc_stems)}")
# P=원문에 그대로 / R=순서 바뀜 / M=일부만 / U=아예 없음

convolutional neural network        -> P
neural networks                     -> P
deep learning                       -> U


### 단일 평가기 — F1@K (K-분모 규칙)

`precision_recall_f1_at_k`: 예측 top-K와 gold를 stem 정규화 후 매칭. present/absent, PRMU recall,
MAP/nDCG도 같은 모듈에서 나온다. 수식 정의: [`../docs/METRICS.md`](../docs/METRICS.md).

In [4]:
from src.metrics import precision_recall_f1_at_k, evaluate_corpus

preds = ["neural network", "deep learning", "image classification", "svm", "cnn"]
gold  = ["neural network", "cnn", "medical imaging"]
r = precision_recall_f1_at_k(preds, gold, 5)
print("P@5 =", round(r["precision"], 3), " R@5 =", round(r["recall"], 3), " F1@5 =", round(r["f1"], 3))

# 코퍼스 단위 매크로 평균 (모든 실험이 이 함수로 최종 점수 산출)
macro = evaluate_corpus([preds], [gold], ks=(5, 10))
print("corpus F1@5 =", round(macro["F1@5"], 3))

P@5 = 0.4  R@5 = 0.667  F1@5 = 0.5
corpus F1@5 = 0.5
